## 1 Upload CSV

In [ ]:
import io, pandas as pd
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
import pandas as pd
import numpy as np

df_loaded = None

u = widgets.FileUpload(accept='.csv', multiple=False)
out = widgets.Output()

def handle_upload(change):
    # Fire only when value changes
    if change['name'] != 'value' or not u.value:
        return
    with out:
        # ipywidgets 8 often returns a tuple of UploadedFile objects
        # older versions return a dict mapping filename -> dict(...)
        try:
            # v8 style: tuple of UploadedFile
            uploaded = u.value[0]
            name = uploaded['name']
            content = uploaded['content']
        except Exception:
            # fallback: dict style
            (name, meta), = u.value.items()
            content = meta['content']
        try:
            global df_loaded
            df_loaded = pd.read_csv(io.BytesIO(content))
        except Exception as e:
            print("Read error:", e)
            return
        print(f"Received: {name} | rows={len(df_loaded)}, cols={df_loaded.shape[1]}")
        display(df_loaded.head())
    # Reset so re-uploading the same file triggers again
    try: u.value = ()
    except Exception: u.value = {}

u.observe(handle_upload, names='value')
display(widgets.VBox([widgets.HTML("<b>Upload CSV:</b>"), u, out]))



## 2 Choose gripper and side

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np

# Define the durations for which you want to predict the workout weight
target_durations = np.array([40, 80, 120, 160, 240])

def duration_to_session(duration):
  if duration < 48:
    return "power"
  elif duration < 82:
    return "power/strength"
  elif duration < 129:
    return "strength"
  elif duration < 180:
    return "strength/endurance"
  else:
    return "endurance"

# Define the desired order of workout types
workout_order = ["power", "power/strength", "strength", "strength/endurance", "endurance"]

df_loaded["workout_type"] = df_loaded["max_hold"].map(duration_to_session)
# Convert 'workout_type' to a categorical type with a specified order
df_loaded["workout_type"] = pd.Categorical(df_loaded["workout_type"], categories=workout_order, ordered=True)

# Group and unstack, the order will be preserved due to the categorical type
workout_counts = df_loaded.groupby(["gripper", "side", "workout_type"], observed=True).size().unstack(fill_value=0)

display(workout_counts)

# Get the workout type counts from the previous cell's output
# The workout_counts DataFrame already has the correct order due to the categorical type

# Get unique grippers
grippers = workout_counts.index.get_level_values('gripper').unique()

# for gripper in grippers:
#     fig = go.Figure()
#     gripper_data = workout_counts.loc[gripper]

#     for side in gripper_data.index:
#         fig.add_trace(go.Bar(
#             x=gripper_data.columns,
#             y=gripper_data.loc[side],
#             name=side
#         ))

#     fig.update_layout(
#         barmode='group',
#         title=f'Workout Type Distribution for {gripper} Gripper',
#         xaxis_title='Workout Type',
#         yaxis_title='Count'
#     )
#     fig.show()

# Dropdowns oder RadioButtons
field_gripper = widgets.RadioButtons(
    options=['micro', 'crusher'],
    description='Gripper:'
)
field_side = widgets.RadioButtons(
    options=['left', 'right'],
    description='Side:'
)
field_unit = widgets.RadioButtons(
    options=['kg', 'lbs'],
    description='Unit:'
)

def data_prep(gripper, side):
  df = df_loaded[(df_loaded["gripper"] == gripper)&(df_loaded["side"]==side)].copy()
  df.loc[:,"transformed_weight"] = df["weight"]
  if field_unit.value == "kg":
    df.loc[:,"transformed_weight"] = df["weight"] * 0.453592

  df.loc[:,'max_hold'] = pd.to_numeric(df['max_hold'], errors='coerce')
  df.loc[:,'transformed_weight'] = pd.to_numeric(df['transformed_weight'], errors='coerce')
  print(f"Selected: {gripper}, {side}: {len(df)} sets")
  return df

df = data_prep("micro", "left")

def on_change(change):
    if change['name'] == 'value' and change['type'] == 'change':
        global df
        df = data_prep(field_gripper.value, field_side.value)

field_gripper.observe(on_change)
field_side.observe(on_change)

display(field_gripper, field_side, field_unit)


## 3. Curve fitting
### Spliced Linear–Exponential fit with bootstrap CIs and proportional-noise PIs
**Purpose.**
This routine fits a performance curve that combines two parts:
1) A **linear power range** and
2) An **exponential endurance range**.

It also visualizes uncertainty around the fitted curve:
* **Confidence bands (CI):** show how the *true underlying power curve* might vary.
* **Prediction bands (PI):** show how *future set outcomes* might spread around the curve.
The prediction bands use **proportional Gaussian noise**, meaning the scatter increases with the weight lifted.

---
### Model (`spliced_linear_exp_model`)
* For durations `x ≤ x1`: a **linear** section with slope `s = (w1 - w0) / x1`
  → `y = w0 + s·x`
* For durations `x > x1`: an **exponential decay** approaching an endurance floor `ef`
  → `y = ef + (w1 - ef) · exp(- (x - x1) / τ)`
  Both parts join smoothly at `(x1, w1)` with the same slope `s`.

---
### Parameters fitted (4)
* **w0** — weight at duration 0 s (approx. 1RM / max power)
* **w1** — weight at splice point `x1` (end of power range)
* **ef** — endurance floor (lowest sustainable weight for very long durations)
* **x1** — splice point in seconds (where power transitions to endurance)

---
### Interpretation
* **Green line:** best-fit curve combining power and endurance behavior
* **Blue band (CI):** uncertainty in the *true* curve (from bootstrap resampling)
* **Orange band (PI):** expected variation in *actual observed sets* (includes random day-to-day fluctuations)


In [ ]:
from scipy.optimize import curve_fit
import plotly.graph_objects as go
import numpy as np # Import numpy

mask = np.isfinite(df['max_hold']) & np.isfinite(df['transformed_weight'])
df_fit = df.loc[mask]
x = df_fit['max_hold'].to_numpy(dtype=float)
y = df_fit['transformed_weight'].to_numpy(dtype=float)


def spliced_linear_exp_model(x: np.ndarray, w0: float, w1: float, ef: float, x1: float) -> np.ndarray:
    """
    :param x: durations
    :param w0: weight at duration 0
    :param w1: weight at slice point x1 (end of linear power section)
    :param ef: endurance floor (asymptotic weight as duration → ∞)
    :param x1: splice point (seconds)
    """
    s = (w1 - w0) / x1
    Ctot = max(1e-9, w1 - ef)
    tau = -Ctot / s
    x = np.asarray(x, dtype=float)
    left = w0 + s * x
    t = np.maximum(0.0, x - x1)
    right = ef + Ctot * np.exp(-t / tau)
    return np.where(x <= x1, left, right)

def initial_guess(x, y):
    # Initial guesses
    min_y = float(y.min())
    max_y = float(y.max())

    x1_init = 80.0
    w0_init = max_y
    w1_init = y.mean()
    ef_init = min_y
    lower_bounds = [min_y, min_y, min_y * 0.1, 30.0]
    upper_bounds = [max_y * 10.0, max_y, max_y, 180.0]
    p0 = np.array([float(w0_init), float(w1_init), float(ef_init), float(x1_init)], dtype=float)
    p0 = np.clip(p0, np.array(lower_bounds, dtype=float), np.array(upper_bounds, dtype=float))

    return lower_bounds, upper_bounds, p0


lower_bounds, upper_bounds, p0 = initial_guess(x, y)

params, cov = curve_fit(
    spliced_linear_exp_model,
    x,
    y,
    p0=p0,
    bounds=(lower_bounds, upper_bounds),
    maxfev=20000,
)
stderr = np.sqrt(np.diag(cov))
w0, w1, ef, x1 = params
y_fit = spliced_linear_exp_model(x, *params)
ss_res = float(np.sum((y - y_fit) ** 2))
ss_tot = float(np.sum((y - y.mean()) ** 2))
r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')
print(
    f"[1b] 1RM: {w0:.4f} ± {stderr[0]:.4f}, strength={w1:.4f} ± {stderr[1]:.4f}, "
    f"endurance floor: {ef:.4f} ± {stderr[2]:.4f}, x1={x1:.2f} ± {stderr[3]:.2f}, R²={r2:.4f}"
)

# Overlay fitted curve
x_line = np.linspace(0.0, 300, 301)
y_line = spliced_linear_exp_model(x_line, *params)

# Bootstrap percentile bands by refitting on resampled (x,y) using model 1b
rng = np.random.default_rng(0)
n_boot = 100
preds = []
for _ in range(n_boot):
    idx = rng.integers(0, len(x), size=len(x))
    xb = x[idx]
    yb = y[idx]

    lower_bounds_b, upper_bounds_b, p0_b = initial_guess(xb, yb)


    try:
        params_b, _ = curve_fit(
            spliced_linear_exp_model,
            xb,
            yb,
            p0=p0_b,
            bounds=(lower_bounds_b, upper_bounds_b),
            maxfev=20000,
        )
        preds.append(spliced_linear_exp_model(x_line, *params_b))
    except Exception:
        continue

y_line_boot_base = y_line
# Bootstrap distribution of the mean curve on x_line
Y_mean = np.vstack(preds)  # shape: [n_boot, len(x_line)]

# --- Confidence band (mean) ---
ci_lo, ci_med, ci_hi = np.percentile(Y_mean, [5, 50.0, 95], axis=0)

# --- Proportional Gaussian Prediction band (future observation) ---
# Estimate proportionality c from residuals: sd ≈ c * mean
tiny = 1e-9
residuals = y - y_fit
mask = np.asarray(y_fit) > tiny
if np.any(mask):
    c = np.sqrt(np.mean((residuals[mask] / np.maximum(y_fit[mask], tiny))**2))
else:
    c = 0.0 # if something degenerate happens


# Heteroscedastic noise SD at each (bootstrap, x) point
sigma_x = c * np.maximum(Y_mean, tiny)            # same shape as Y_mean
E = rng.normal(loc=0.0, scale=sigma_x)            # proportional Gaussian noise
Y_pred = Y_mean + E                                # distribution of future obs

pi_lo, pi_med, pi_hi = np.percentile(Y_pred, [5, 50.0, 95], axis=0)

# ---- Plot everything ----
fig_show = go.Figure()
# Data points
fig_show.add_trace(go.Scatter(x=x, y=y, mode='markers', name='data', text=df_fit['date_time']))

# 90% CI (mean) band (using 5th and 95th percentiles for 90% CI) - Keep for reference but set fillcolor to transparent
fig_show.add_trace(go.Scatter(x=x_line, y=ci_lo, mode='lines', name='CI low', line=dict(width=0), showlegend=False, hoverinfo='skip'))
fig_show.add_trace(go.Scatter(x=x_line, y=ci_hi, mode='lines', name='90% CI (force curve)', fill='tonexty', line=dict(width=0), fillcolor='rgba(99,110,250,0.20)'))


# 90% PI (future observation) band (using 5th and 95th percentiles for 90% PI) — plotted behind main curve so it’s visible and wider
fig_show.add_trace(go.Scatter(x=x_line, y=pi_lo, mode='lines', name='PI low', line=dict(width=0), showlegend=False, hoverinfo='skip'))
fig_show.add_trace(go.Scatter(x=x_line, y=pi_hi, mode='lines', name='90% PI (set outcomes)', fill='tonexty', line=dict(width=0), fillcolor='rgba(255,127,14,0.20)'))

# Fitted curve
fig_show.add_trace(go.Scatter(x=x_line, y=y_line, mode='lines', name='Curve fit', line=dict(color='rgba(0,204,150,1)')))

# # Add vertical lines at target_durations
# for duration in target_durations:
#     fig_show.add_shape(
#         type="line",
#         x0=duration,
#         y0=0,
#         x1=duration,
#         y1=max(y), # Extend line to max y value or beyond if needed
#         line=dict(
#             color="Grey",
#             width=1,
#             dash="dot",
#         )
#     )


fig_show.update_layout(xaxis_title='duration (s)', yaxis_title='weight', legend_title_text='')
fig_show.show()

## 3. Workout weight recommendations

In [ ]:
# Use the fitted model to predict the workout weight for the given durations
recommended_weights = spliced_linear_exp_model(target_durations, *params)

# Interpolate the bootstrapped prediction intervals to the desired durations
# We need to use the x_line and the pi_lo, pi_hi from cell t_qWTb0-7kbo
# Make sure x_line is sorted
sorted_indices = np.argsort(x_line)
x_line_sorted = x_line[sorted_indices]
pi_lo_sorted = pi_lo[sorted_indices]
pi_hi_sorted = pi_hi[sorted_indices]

# Interpolate the lower and upper bounds at the desired durations
recommended_lo = np.interp(target_durations, x_line_sorted, pi_lo_sorted)
recommended_hi = np.interp(target_durations, x_line_sorted, pi_hi_sorted)

# Round the bounds to the nearest 0.25 kg (lower bound down, upper bound up)
recommended_lo_rounded = np.round(recommended_lo * 4) / 4
recommended_hi_rounded = np.round(recommended_hi * 4) / 4

# Print the recommended weights with the estimated range in parentheses and relative percentage
print("Recommended workout weights with estimated range:")
for duration, weight, lower_bound, upper_bound in zip(target_durations, recommended_weights, recommended_lo_rounded, recommended_hi_rounded):
    print(f"{duration}s: {np.round(weight*4)/4:.2f}{field_unit.value} ({lower_bound:.2f}-{upper_bound:.2f}{field_unit.value})")